# Dunnhumby NGCF · 학교 계정 seed50·51 전용

**개인7시드 배정 확인서가 있어야 시작합니다.** 42~48·49 학습을 차단하며 완료 결과는 재학습/재평가하지 않습니다.

고정 조건은 변경하지 않습니다: NGCF 8모형, 300epoch 마지막점 1회 test,
DAY1~704 학습/DAY705~711 신규상품 추천, k1, uniform K1, batch8192,
Adam .0005, M2/M3/M4 공동학습, L4. validation/holdout/조기종료/중간test 없음.
최종 계획은8모형×10시드이며 역할별 완료는 전체80개 완료와 다릅니다.

저장은 학교 Drive입니다. 사용자 승인으로 시작 기준은1GB, 매 저장 전에는1GB+현재 저장 요청량의 실제 여유를 검사합니다. 부족하면 멈춥니다. 각 모형 최신2epoch의 **모델·Adam·RNG 전체 상태**를 영구 보존합니다. 새 실행 전용 폴더의 더 오래된 checkpoint만 자동 영구 삭제하며 기존 실험/휴지통 전체를 정리하지 않습니다. 새epoch는 별도 파일로 업로드하고 크기·MD5 확인 후 다음epoch로 진행합니다. Colab 로컬만 저장하고 넘어가지 않습니다. 다른 실행의 미래 사용량까지 보장하지 않으므로 외부 저장증가 시 안전 중단할 수 있습니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), '런타임 유형을 L4 GPU로 설정하세요.'
GPU = torch.cuda.get_device_name(0)
assert 'L4' in GPU, f'현재 GPU: {GPU}. 기존 조건과 같은 L4로 실행하세요.'
SOURCE_COMMIT = '63146eff4ed60bfb0cc430b77d58fdc40e4db7c5'
REPO = Path('/content/clv-dh-ngcf-remaining9-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == SOURCE_COMMIT
sys.path.insert(0, str(REPO))
print('GPU:', GPU, '| 고정 코드:', SOURCE_COMMIT)
import hashlib, urllib.request, importlib.util
HELPER_COMMIT = 'b38ba01d1df480db30284075c09363570edfbd11'
HELPERS = {
    'clv_dh_ngcf_split72': '96c0b3d9a08d2a244c1bc15ea1967fa3414b85f46754cdb69ad9bfeba2b5fb43',
    'clv_ngcf_school_storage': '31c3a6abed6b8ab72fb94f0de6b739e53cb8a6dd828469f2aba84ea620e793db',
}
for name, expected in HELPERS.items():
    url = f'https://raw.githubusercontent.com/jung-un/clv-m2-lightgcn-runner/{HELPER_COMMIT}/{name}.py'
    with urllib.request.urlopen(url, timeout=60) as response:
        data = response.read()
    assert hashlib.sha256(data).hexdigest() == expected
    target = Path('/content') / (name + '.py')
    target.write_bytes(data)
    spec = importlib.util.spec_from_file_location(name, target)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
import clv_dh_ngcf_split72 as split
import clv_ngcf_school_storage as school_storage
school_storage.START_FREE = school_storage.MIN_FREE = 1_000_000_000  # 사용자 지정: 1GB 여유 유지
print('고정 학습코드63146eff / 분담도우미:', HELPER_COMMIT)


## 1. 학교 Drive 계정·실제 저장 여유 확인
계정 인증은 학교 계정으로 합니다. 기존 다른 실험은 중단하지 않습니다.

In [ ]:
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
credentials, _ = google.auth.default()
drive_api = build('drive', 'v3', credentials=credentials, cache_discovery=False)
print('실제 여유 GB:', school_storage.check_space(drive_api, initial=True)/1e9)


## 2. 고정 입력 묶음·개인 배정 확인서 준비
입력은 압축 상태만Drive에 두고 실제 압축해제는/content에서 합니다. 파일을 못 찾으면 두 파일을 업로드하세요.

In [ ]:
from google.colab import files
import shutil, zipfile
INPUT_SHA = '7e71203b53da9b06eb94266ae48fa8e64fedf215198a32e96de91527da889b72'
bundle = Path('/content/drive/MyDrive/dh_ngcf_school2_inputs.zip')
receipt_path = Path('/content/drive/MyDrive/dh_ngcf_personal7_assignment.json')
if not bundle.is_file() or not receipt_path.is_file():
    print('Mac의 dh_inputs.zip 및 개인 배정 확인서 dh_ngcf_personal7_assignment.json을 선택하세요.')
    uploaded = files.upload()
    for name in uploaded:
        source = Path(name)
        if source.name == 'dh_inputs.zip':
            assert hashlib.sha256(source.read_bytes()).hexdigest() == INPUT_SHA
            if not bundle.exists(): shutil.copy2(source, bundle)
        elif source.name == 'dh_ngcf_personal7_assignment.json':
            if receipt_path.exists():
                assert receipt_path.read_bytes() == source.read_bytes(), '다른 배정 확인서: 자동 교체 금지'
            else: shutil.copy2(source, receipt_path)
assert bundle.is_file() and receipt_path.is_file()
assert hashlib.sha256(bundle.read_bytes()).hexdigest() == INPUT_SHA
inputs = Path('/content/ngcf_school2_inputs')
with zipfile.ZipFile(bundle) as z:
    assert z.testzip() is None
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(inputs)
expected = json.loads((inputs / 'manifest.json').read_text())
for name, sha in expected.items():
    assert hashlib.sha256((inputs / name).read_bytes()).hexdigest() == sha, name
print('입력·기존 seed49 참조 해시 확인 완료. 원본은 학습에 재사용합니다.')


## 3. 동일 입력·설정 검증 및 저장 설치 — 학습 없음
입력/특징/M3·M4 진단은 원래seed49와 대조하고, 개인 실행과도 학습 조건 지문을 대조합니다. 경로 차이는 기록하되 결과 신원은 계정별로 보존합니다.

In [ ]:
import clv_dh_ngcf_remaining9 as screen
assert Path(screen.__file__).resolve().parent == REPO.resolve()
assert Path(screen.original.__file__).resolve().parent == REPO.resolve()
screen.base.v3.SCHEMA['dunnhumby']['tx_path'] = str(inputs / 'raw/dunnhumby/transaction_data.csv')
screen.base.v3.SCHEMA['dunnhumby']['item_meta_path'] = str(inputs / 'raw/dunnhumby/product.csv')
cfg = screen.configure(
    out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_ngcf_school2_seed50_51_v1',
    m2_reference=str(inputs / 'references/clv_m2_adaptive_dunnhumby_final10_results.zip'),
    m4_reference=str(inputs / 'references/clv_m4_user_centered_final10_results.zip'),
    seed49_reference=str(inputs / 'references/dh_ngcf_final8_seed49_results.zip'),
)
prepared = screen.prepare(cfg)
split.check_personal_receipt(prepared, receipt_path)
split.install(prepared, 'school')
checkpoint_folder = school_storage.install(prepared, drive_api)
print('학교 담당:', split.ASSIGNMENTS['school'])
print('결과/이력:', prepared['run_dir'])
print('영구 체크포인트 폴더:', checkpoint_folder)


## 4. seed50→51, 각8모형 순차 학습
중단 시 이 노트북을 같은 경로로 다시 실행합니다. 영구 checkpoint를 검증·다운로드한 뒤 저장epoch 다음부터 이어갑니다. 완료 모델은 건너뜁니다.

In [ ]:
try:
    result = split.run(prepared, 'school')
finally:
    output = split.export_partition(prepared, 'school',
        '/content/drive/MyDrive/dh_ngcf_school2_results.zip')
    print('학교 완료 결과 패킷:', output)
print(json.dumps(result, ensure_ascii=False, indent=2))
assert result['complete'] and result['completed'] == 16


## 5. 결과 패킷 다운로드
개인7시드 결과와 합산할 ZIP입니다. 큰checkpoint는 별도학교 checkpoint폴더에 남습니다.

In [ ]:
from google.colab import files
files.download(split.export_partition(prepared, 'school', '/content/dh_ngcf_school2_results.zip'))
